In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import numpy as np

In [ ]:
import sys
from pathlib import Path

for p in (Path.cwd(), Path.cwd() / "notebook", Path.cwd().parent / "notebook"):
    if (p / "lm_utils.py").exists():
        sys.path.insert(0, str(p))
        break

from lm_utils import display_generated_name, reset_seed, visualize_probabilities

SEED = 42
LICZBA_EPOK = 12
LEARNING_RATE = 0.003
TEMPERATURA = 0.4
EMBEDDING_DIM = 64
GRU_UNITS = 128
DROPOUT = 0.2

reset_seed(SEED)

W tym projekcie zajmiemy się technologią, która jest bardzo popularna i przydatna oraz jest jednym z głównych komponentów takich modeli jak ChatGPT - *Language Model* : [Wikipedia](https://en.wikipedia.org/wiki/Language_model)<br>

W dużym uproszczeniu staramy się wypredykować kolejne słowo w danej sekwencji. Pomyślcie o następującym zdaniu:<br><br>
***Mamy dzisiaj bardzo słoneczną ...***<br><br>
Zapewne zdecydowana większość z was zaproponowałaby kończące słowo $pogodę$, być może ktoś zaproponowałby słowo $atmosferę$, ale prawie na pewno nikt nie zaproponowałby słowa $krzesło$.<br>
Upraszczając właśnie takie zadanie, taką predykcję, wykonują modele takie jak ChatGPT (oczywiście pozwalają one również zadać pytanie, które bardziej ogólnie nazywamy promptem). Zatem nasz model ML'owy ma za zadanie wyestymować warunkowy rozkład słów (czyli prawdopodobieństwa dla poszczególnych słów, wszystkich słów w słowniku) pod warunkiem tej niedokończonej sekwencji:<br><br>
```
P('pogodę' | ['Mamy', 'dzisiaj', 'bardzo', 'słoneczną']) = 0.95
P('atmosferę' | ['Mamy', 'dzisiaj', 'bardzo', 'słoneczną']) = 0.03
P('krzesło' | ['Mamy', 'dzisiaj', 'bardzo', 'słoneczną']) ~ 0
```
**Oczywiście gdybyśmy zsumowali te prawdopodobieństwa warunkowe po wszystkich słowach w słowniku to otrzymalibyśmy wartość 1!**<br><br>

Zwykle taki *Language Model* musi być uczony na ogromnych korpusach danych, liczących Terabajty (np. cała wikipedia). My, aby zaprezentować podobny mechanizm, "zejdziemy jedno oczko niżej" - **zamiast predykować następne słowo w zdaniu będziemy predykować następną literę w wyrazie.**<br><br>
Nasz *Language Model* będzie dotyczył "języka" **łacińskich nazw chorób** z repozytorium sympozjum. Za pomocą naszego modelu będziemy mogli generować nowe nazwy chorób!

Dane pochodzą z pliku `data/choroby.csv` w repozytorium [NXTRSS/sympozjum-pediatria](https://github.com/NXTRSS/sympozjum-pediatria).

W poniższej komórce odczytujemy nazwy (najpierw z lokalnego pliku, w razie braku — z internetu):

In [ ]:
REPO = "https://raw.githubusercontent.com/NXTRSS/sympozjum-pediatria/main/data/choroby.csv"

try:
    df = pd.read_csv("../data/choroby.csv", header=None, names=["nazwa"], sep="\t")
except FileNotFoundError:
    try:
        df = pd.read_csv("data/choroby.csv", header=None, names=["nazwa"], sep="\t")
    except FileNotFoundError:
        print("Plik lokalny nie znaleziony. Pobieram dane z internetu...")
        df = pd.read_csv(REPO, header=None, names=["nazwa"], sep="\t")

df.head()

In [ ]:
df.shape

In [ ]:
list_of_names = (
    df["nazwa"].dropna().astype(str).str.strip().str.lower().drop_duplicates().tolist()
)
rng = np.random.default_rng(SEED)
list_of_names = [list_of_names[i] for i in rng.permutation(len(list_of_names))]
list_of_names[:10]

Każdą nazwę poprzedzamy tokenem sos (start of sentence) i kończymy eos (end of sentence):<br>
```
<sos>pneumonia</eos>
```
W naszym przypadku umawiamy się na `%` i `!` — ponieważ model będzie oparty o pojedyncze litery i w takiej sytuacji jest wygodniej gdy specjalne tokeny również są pojedynczymi znakami.<br>
```
%pneumonia!
```

### Zadanie 1 — dodaj tokeny do każdej nazwy

Dla każdej nazwy w `list_of_names` utwórz nowy string poprzez dodanie `%` na początku i `!` na końcu. Wynik zapisz do listy `name_list`. Polecam użyć *list comprehension*.

###### <span style="color: #c17f24;">💡 Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

Konkatenacja stringów w Pythonie: `'%' + tekst + '!'` daje `'%tekst!'`.

*List comprehension* — wzorzec: `[wyrażenie for x in lista]`.

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
name_list = ['%' + name + '!' for name in list_of_names]

name_list[:10]

###### <span style="color: #4a7090;">📊 Spodziewany wynik</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

```
['%abacterial ventriculitis!', '%abarticular!', '%abarthrosis!', ...]
```

(kolejność zależy od tasowania ziarnem 42 — na sali zawsze ta sama lista).

######

No dobrze — mamy gotowy zestaw danych, nasz "język" nazw chorób. Teraz musimy pomyśleć nad modelem, który będzie estymował rozkład warunkowy poszczególnych liter/znaków. Do tego celu wykorzystamy sieci RNN.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Embedding, GRU, Dense
from tensorflow.keras.losses import SparseCategoricalCrossentropy
import tensorflow.keras as keras

In [ ]:
# resetujemy filtr aby nie usunął tokenów % i !
tokenizer = Tokenizer(filters='', lower=True, char_level=True)
tokenizer.fit_on_texts(name_list)

vocab_size = len(tokenizer.word_index) + 1
print(f'Wielkość naszego słownika wynosi: {vocab_size}')

Możemy zobaczyć kilka elementów z naszego słownika:

In [ ]:
Nth = ['Pierwszy','Drugi','Trzeci', 'Czwarty', 'Piąty', 'Szósty', 'Siódmy', 'Ósmy', 'Dziewiąty']
for nth, items in zip(Nth, tokenizer.word_index.items()):
    print(f'{nth:8} - {items[0]}:{items[1]}')

## Czym jest X i Y w naszym zadaniu?

Dla nazwy `%pneumonia!` tworzymy parę:
```
      X      ->      Y
%pneumonia -> pneumonia!
```
Model RNN będzie tak "sprytny", że na podstawie takiego X i Y będzie liczył po kolei:
```
1. P(p | %)
2. P(n | %p)
...
```

## Padding — wyrównanie długości sekwencji

Nazwy mają różne długości. Uzupełniamy krótsze zerami (padding), a podczas uczenia maskujemy te zera (`mask_zero=True`), żeby nie zaniżały straty.

In [ ]:
X_train = [name[:-1] for name in name_list]
y_train = [name[1:] for name in name_list]

przyklad = list_of_names[0]
idx = list_of_names.index(przyklad) if przyklad in list_of_names else 0
print(f'Przykładowe wejście:  {X_train[idx]}')
print(f'Przykładowe wyjście: {y_train[idx]}')

X_train = tokenizer.texts_to_sequences(X_train)
y_train = tokenizer.texts_to_sequences(y_train)
X_train = pad_sequences(X_train, padding='post')
y_train = pad_sequences(y_train, padding='post')

print(f'Kształt X_train: {X_train.shape}')

## Konstrukcja modelu

Zbudujemy klasę `LanguageModel` dziedziczącą po `keras.Model`. Składa się z trzech warstw:

1. **`Embedding`** — zamienia każdy token na gęsty wektor o wymiarze **64**. `mask_zero=True` maskuje padding.
2. **`GRU`** — sieć rekurencyjna, **128** jednostek, **dropout 0.2**. Zwraca wyjście dla każdego kroku i końcowy hidden state.
3. **`Dense`** — warstwa wyjściowa z `softmax` nad słownikiem.

> Ustawienia sympozjum (po przeszukaniu hiperparametrów): 12 epok, Adam `lr=0.003`, temperatura **0.4** przy losowaniu.

### Zadanie 2 — uzupełnij klasę LanguageModel

Uzupełnij metodę `__init__`. Trzy warstwy: `Embedding`, `GRU`, `Dense`. Pamiętaj o `mask_zero`, `return_sequences`, `return_state`, `dropout`.

###### <span style="color: #c17f24;">💡 Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

**Embedding:** `vocab_size`, `output_dim=64` (sympozjum), `mask_zero=True`.

**GRU:** `128` jednostek, `dropout=0.2`, `return_sequences=True`, `return_state=True`.

**Dense:** `vocab_size`, `activation='softmax'`.

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
reset_seed(SEED)

class LanguageModel(Model):
    def __init__(self):
        super().__init__()
        self.embedding = Embedding(vocab_size, output_dim=EMBEDDING_DIM, mask_zero=True)
        self.rnn = GRU(GRU_UNITS, return_sequences=True, return_state=True, dropout=DROPOUT)
        self.fc = Dense(vocab_size, activation='softmax')

    def call(self, inputs, training=False, hidden_state=None, return_state=False):
        x = self.embedding(inputs, training=training)
        seq_out, last_hidden_state = self.rnn(x, training=training, initial_state=hidden_state)
        x = self.fc(seq_out)
        if return_state:
            return x, last_hidden_state
        return x


model = LanguageModel()
model(X_train)
model.summary()

######

### Zadanie 3 — skompiluj i wytrenuj model

Zdefiniuj funkcję straty, skompiluj model z **Adam**(learning_rate=**0.003**) i wytrenuj przez **12** epok (batch 64).

###### <span style="color: #c17f24;">💡 Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

**Funkcja straty:** `SparseCategoricalCrossentropy()`.

**Kompilacja:** `optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE)`.

**Trening:** `model.fit(X_train, y_train, epochs=LICZBA_EPOK, batch_size=64)`.

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
reset_seed(SEED)

criterion = SparseCategoricalCrossentropy()
model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss=criterion)
history = model.fit(X_train, y_train, epochs=LICZBA_EPOK, batch_size=64, verbose=2)
historia = history  # alias pod testy CI

######

##### Wyrysujmy jak zachowywał się błąd w trakcie uczenia — czyli przez epoki

In [ ]:
loss = history.history['loss']

plt.figure(figsize=(6, 4))
plt.plot(loss, label='Błąd uczenia', color='blue')
plt.xlabel('Epoki')
plt.ylabel('Błąd')
plt.title('Błąd uczenia w trakcie epok')
plt.legend()
plt.grid(True)
plt.show()

## Generowanie nowych nazw chorób

Model jest generatywny — potrafi tworzyć "nowe" obiekty. W każdym kroku:
- przekazujemy tylko **ostatnio wygenerowany znak** + **hidden state** z poprzedniego kroku,
- model zwraca rozkład prawdopodobieństwa dla kolejnego znaku,
- wybieramy znak: deterministycznie (`greedy=True`) lub losowo z rozkładu po **temperaturze** `TEMPERATURA` (u nas **0.4**).

Wizualizacja kroków (`visualize_probabilities`) jest w module `lm_utils.py` — ten sam wykres słupkowy co w notatniku dinozaurów.

In [ ]:
def sample_model_optimal(
    prompt='%',
    greedy=False,
    temperature=TEMPERATURA,
    print_outcome=True,
    visuals=False,
    known_names=None,
):
    prompt = prompt.lower()
    encoded = tokenizer.texts_to_sequences([prompt])
    encoded = tf.convert_to_tensor(encoded)
    model_out, hidden_state = model(encoded, return_state=True)
    last_output = model_out[0, -1, :].numpy()

    if visuals:
        visualize_probabilities(last_output, 0, tokenizer, prompt)

    for i in range(30):
        last_output = np.array(last_output, dtype=np.float64)
        last_output[0] = 0.0
        s = last_output.sum()
        if s > 0:
            last_output = last_output / s

        if greedy:
            next_char_idx = int(last_output.argmax())
        else:
            log_p = np.log(last_output + 1e-9) / max(temperature, 1e-6)
            last_output = np.exp(log_p)
            last_output[0] = 0.0
            last_output = last_output / last_output.sum()
            next_char_idx = int(np.random.choice(np.arange(len(last_output)), p=last_output))

        next_char = tokenizer.index_word[next_char_idx]
        prompt += next_char

        if next_char == '!':
            break

        encoded = tokenizer.texts_to_sequences([next_char])
        encoded = tf.convert_to_tensor(encoded)
        model_out, hidden_state = model(encoded, hidden_state=hidden_state, return_state=True)
        last_output = model_out[0, -1, :].numpy()

        if visuals:
            visualize_probabilities(last_output, i + 1, tokenizer, prompt)

    if print_outcome:
        nazwa = prompt[1:-1]
        display(Markdown(f"## 🩺 **Wygenerowana nazwa choroby:** `{nazwa}`"))
        display_generated_name(prompt, known_names=known_names)
    if visuals:
        plt.show()
    return prompt

In [ ]:
reset_seed(SEED)
prompt = '%'
generated_text = sample_model_optimal(
    prompt, visuals=True, temperature=TEMPERATURA, known_names=set(list_of_names)
)
pierwsza_nazwa = generated_text[1:-1]

## Dziesięć nazw — prosty test nowości

Jeśli nazwa **jest w zbiorze treningowym**, zobaczysz ⚠️; jeśli **model coś wymyślił** — ✨.

In [ ]:
reset_seed(SEED)
lista_nazw = []
znane = set(list_of_names)
for _ in range(10):
    wynik = sample_model_optimal(
        "%",
        print_outcome=False,
        visuals=False,
        temperature=TEMPERATURA,
        known_names=znane,
    )
    lista_nazw.append(wynik[1:-1])
    display_generated_name(wynik, known_names=znane)